# Trabalho LEA — Bandits e decisões sequenciais

**Exercícios 1 e 2 — notebook para Google Colab**

Este notebook implementa as políticas solicitadas, executa simulações de Monte Carlo, produz tabelas e gráficos e inclui textos de interpretação para o relatório.

Execute as células na ordem. Para testes rápidos, altere `MODO_COMPLETO = False`.

## 0. Bibliotecas e funções auxiliares

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

SEMENTE_GLOBAL = 770291
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# True: resultados finais; False: apenas teste rápido
MODO_COMPLETO = True

# Análises extras de sensibilidade podem aumentar bastante o tempo de execução.
EXECUTAR_ANALISES_EXTRAS = False


def argmax_aleatorio(valores, gerador):
    """Escolhe aleatoriamente entre os índices empatados no maior valor."""
    valores = np.asarray(valores)
    candidatos = np.flatnonzero(np.isclose(valores, np.max(valores)))
    return int(gerador.choice(candidatos))


# Cores fixas: a mesma política terá a mesma cor em todos os gráficos.
CORES_POLITICAS = {
    "Aleatória": "C0",
    "Greedy": "C1",
    "Thompson Sampling": "C2",
    "UCB": "C3",
    "Epsilon-Greedy": "C4",
}

print("Ambiente preparado.")

# Parte 1 — Inferência após seleção

Nesta parte, o objetivo **não** é descobrir qual política obtém maior recompensa. O exercício foi construído para investigar um problema de inferência: o que acontece quando usamos os dados para escolher o braço “vencedor” e, depois, construímos um intervalo de confiança usual para esse mesmo braço como se ele tivesse sido escolhido antes?

Para cada braço,

$$
R_t \mid a_t=a \sim N(\mu_a,1),
\qquad
\mu_1=\cdots=\mu_5=0.
$$

Portanto, no cenário simulado, **nenhum braço é realmente melhor que os outros**. Qualquer braço que pareça melhor ao final do experimento parece melhor apenas por flutuação amostral.

Ao final de cada experimento, selecionamos

$$
\widehat a = \arg\max_{a\in\{1,\ldots,5\}}\widehat\mu_a
$$

e construímos o intervalo usual

$$
\widehat\mu_{\widehat a}
\pm
1{,}96\frac{1}{\sqrt{n_{\widehat a}}}.
$$

Esse intervalo é chamado aqui de **ingênuo**, porque ignora que o braço foi escolhido justamente por ter apresentado a maior média observada. A simulação estima a cobertura empírica desse intervalo, isto é, a proporção de vezes em que o intervalo contém o valor verdadeiro $0$.

## 1.1 Escolhas operacionais para implementar as políticas

O enunciado da Parte 1 fixa o modelo das recompensas, mas não especifica uma priori para as médias $\mu_a$. Para implementar Thompson Sampling e uma versão Bayesiana do UCB, precisamos representar a incerteza sobre cada média ao longo do experimento.

Por simplicidade, usamos prioris independentes

$$
\mu_a\sim N(m_0,\tau_0^2),
\qquad
m_0=0,
\quad
\tau_0=1.
$$

Essa é uma escolha operacional do notebook, não uma hipótese adicional dada no enunciado. Ela é conveniente porque o enunciado já fixa a variância observacional como conhecida, $\sigma^2=1$. Assim, após $n_a$ observações no braço $a$, com soma $S_a$, a posteriori de $\mu_a$ continua Normal, com

$$
v_a=
\left(
\tau_0^{-2}
+
n_a\sigma^{-2}
\right)^{-1},
\qquad
m_a=
v_a
\left(
m_0\tau_0^{-2}
+
S_a\sigma^{-2}
\right).
$$

A inferência final avaliada pelo exercício, porém, **não** é Bayesiana. Depois da coleta, seguimos exatamente o enunciado: calculamos as médias amostrais, escolhemos o braço de maior média e construímos o intervalo usual

$$
\widehat\mu_{\widehat a}
\pm
1{,}96/\sqrt{n_{\widehat a}}.
$$

Também avaliamos, como diagnóstico adicional, a cobertura de um braço fixado previamente. Esse braço fixo não é pedido no enunciado; ele serve como controle para mostrar que o intervalo usual funciona quando o braço é definido antes, mas pode perder cobertura quando escolhemos o vencedor após observar os dados.

## 1.2 Políticas comparadas na Parte 1

As quatro políticas diferem apenas na forma como escolhem o braço em cada instante $t$.

- **Alocação uniforme aleatória:** em cada rodada, escolhe um dos cinco braços com probabilidade $1/5$. Essa política não usa as recompensas observadas para adaptar a alocação.

- **Greedy:** escolhe o braço com maior média amostral observada até o momento. Em caso de empate, o desempate é aleatório. Essa política explora pouco, pois tende a insistir rapidamente no braço que parece melhor no início.

- **UCB:** escolhe o braço com maior escore

  $$
  \text{escore}_a(t)
  =
  m_a(t)
  +
  c\sqrt{\log(t+1)}\sqrt{v_a(t)}.
  $$

  Aqui, $m_a(t)$ e $v_a(t)$ são a média e a variância posteriores de $\mu_a$. Usamos $c=2$, valor usual em simulações didáticas, para dar peso explícito à incerteza. O primeiro termo favorece braços com média estimada alta; o segundo favorece braços ainda incertos.

- **Thompson Sampling:** para cada braço, sorteia

  $$
  \widetilde\mu_a \sim N(m_a(t),v_a(t))
  $$

  e escolhe o braço com maior valor sorteado. Assim, braços que parecem bons são escolhidos com frequência, mas braços incertos ainda têm chance de serem explorados.

Todos os braços recebem uma observação inicial para que as médias amostrais estejam definidas. Essa inicialização também evita que greedy e UCB sejam determinados apenas por empates no instante inicial.

In [ ]:
# Parâmetros da Parte 1
K1 = 5
MU_VERDADEIRO = np.zeros(K1)
SIGMA = 1.0
T1 = 500
M0 = 0.0
TAU0 = 1.0
C_UCB1 = 2.0
POLITICAS1 = ["Aleatória", "Greedy", "UCB", "Thompson Sampling"]
BRACO_FIXO = 0  # braço 1, definido antes da observação dos dados


def posterior_normal(contagens, somas, m0=M0, tau0=TAU0, sigma=SIGMA):
    precisao = 1 / tau0**2 + contagens / sigma**2
    variancia = 1 / precisao
    media = variancia * (m0 / tau0**2 + somas / sigma**2)
    return media, variancia


def simular_gaussiano(politica, semente, horizonte=T1):
    gerador = np.random.default_rng(semente)
    contagens = np.zeros(K1, dtype=int)
    somas = np.zeros(K1)

    # Uma observação inicial por braço garante médias amostrais definidas.
    for a in range(K1):
        r = gerador.normal(MU_VERDADEIRO[a], SIGMA)
        contagens[a] += 1
        somas[a] += r

    for t in range(K1, horizonte):
        medias_amostrais = somas / contagens
        medias_post, vars_post = posterior_normal(contagens, somas)

        if politica == "Aleatória":
            a = int(gerador.integers(K1))
        elif politica == "Greedy":
            a = argmax_aleatorio(medias_amostrais, gerador)
        elif politica == "UCB":
            beta_t = C_UCB1 * np.sqrt(np.log(t + 1))
            a = argmax_aleatorio(
                medias_post + beta_t * np.sqrt(vars_post),
                gerador
            )
        elif politica == "Thompson Sampling":
            amostras = gerador.normal(medias_post, np.sqrt(vars_post))
            a = argmax_aleatorio(amostras, gerador)
        else:
            raise ValueError("Política desconhecida")

        r = gerador.normal(MU_VERDADEIRO[a], SIGMA)
        contagens[a] += 1
        somas[a] += r

    medias = somas / contagens

    # Inferência após seleção: escolhemos o braço com maior média.
    a_sel = argmax_aleatorio(medias, gerador)
    media_sel = medias[a_sel]
    n_sel = contagens[a_sel]
    margem_sel = 1.96 * SIGMA / np.sqrt(n_sel)
    li_sel, ls_sel = media_sel - margem_sel, media_sel + margem_sel

    # Controle: braço definido antes da coleta.
    media_fixa = medias[BRACO_FIXO]
    n_fixo = contagens[BRACO_FIXO]
    margem_fixa = 1.96 * SIGMA / np.sqrt(n_fixo)
    li_fixo = media_fixa - margem_fixa
    ls_fixo = media_fixa + margem_fixa

    contagens_ordenadas = np.sort(contagens)[::-1]
    n_mais_alocado = contagens_ordenadas[0]

    return {
        "politica": politica,
        "braco_selecionado": a_sel + 1,
        "media_selecionada": media_sel,
        "n_selecionado": n_sel,
        "limite_inferior": li_sel,
        "limite_superior": ls_sel,
        "cobriu": li_sel <= 0 <= ls_sel,
        "media_braco_fixo": media_fixa,
        "n_braco_fixo": n_fixo,
        "cobriu_braco_fixo": li_fixo <= 0 <= ls_fixo,
        "n_mais_alocado": n_mais_alocado,
        "proporcao_mais_alocada": n_mais_alocado / horizonte,
        "contagens": contagens,
        "contagens_ordenadas": contagens_ordenadas,
    }


# Teste simples
print(simular_gaussiano("Thompson Sampling", SEMENTE_GLOBAL))

## 1.2 Simulação de Monte Carlo

In [ ]:
B1 = 5000 if MODO_COMPLETO else 300
linhas1 = []

for i, politica in enumerate(POLITICAS1):
    for b in tqdm(range(B1), desc=f"Parte 1 — {politica}"):
        res = simular_gaussiano(
            politica,
            SEMENTE_GLOBAL + 100_000 * i + b
        )

        linha = {
            k: v for k, v in res.items()
            if k not in {"contagens", "contagens_ordenadas"}
        }

        for j, n in enumerate(res["contagens"], 1):
            linha[f"n_braco_{j}"] = n

        for j, n in enumerate(res["contagens_ordenadas"], 1):
            linha[f"n_ordem_{j}"] = n

        linhas1.append(linha)

resultados1 = pd.DataFrame(linhas1)
display(resultados1.head())

In [ ]:
resumo1 = (
    resultados1
    .groupby("politica", sort=False)
    .agg(
        cobertura_empirica=("cobriu", "mean"),
        cobertura_braco_fixo=("cobriu_braco_fixo", "mean"),
        media_da_media_selecionada=("media_selecionada", "mean"),
        desvio_da_media_selecionada=("media_selecionada", "std"),
        mediana_da_media_selecionada=("media_selecionada", "median"),
        media_n_selecionado=("n_selecionado", "mean"),
        media_n_mais_alocado=("n_mais_alocado", "mean"),
        media_proporcao_mais_alocada=("proporcao_mais_alocada", "mean"),
    )
    .reset_index()
)

for coluna, prefixo in [
    ("cobertura_empirica", "selecionado"),
    ("cobertura_braco_fixo", "fixo"),
]:
    ep = np.sqrt(resumo1[coluna] * (1 - resumo1[coluna]) / B1)
    resumo1[f"erro_padrao_MC_{prefixo}"] = ep
    resumo1[f"IC_MC_inferior_{prefixo}"] = resumo1[coluna] - 1.96 * ep
    resumo1[f"IC_MC_superior_{prefixo}"] = resumo1[coluna] + 1.96 * ep

display(resumo1)

## 1.3 Por que comparar braço selecionado e braço fixo?

O enunciado pede a cobertura do intervalo para o braço selecionado, $\widehat a$, isto é, o braço que apresentou a maior média amostral ao final do experimento.

Além disso, calculamos a cobertura para o braço 1, definido antes da observação dos dados. Esse segundo cálculo é apenas um controle metodológico. Ele ajuda a separar dois efeitos:

1. **intervalo aplicado a um braço fixo:** não há seleção do vencedor, então esperamos cobertura próxima de 95%;
2. **intervalo aplicado ao braço selecionado:** há seleção pela maior média, então a cobertura pode ficar abaixo de 95%.

Essa comparação materializa o ponto central do exercício: o problema não é a fórmula do intervalo em si, mas o fato de usá-la depois de escolher o braço mais favorável observado.

In [ ]:
# ------------------------------------------------------------------
# Cobertura: braço selecionado versus braço fixado previamente
# ------------------------------------------------------------------
ordem = POLITICAS1
tab = resumo1.set_index("politica").loc[ordem].reset_index()
x = np.arange(len(ordem))
largura = 0.36

erro_sel = np.vstack([
    tab["cobertura_empirica"] - tab["IC_MC_inferior_selecionado"],
    tab["IC_MC_superior_selecionado"] - tab["cobertura_empirica"],
])

erro_fixo = np.vstack([
    tab["cobertura_braco_fixo"] - tab["IC_MC_inferior_fixo"],
    tab["IC_MC_superior_fixo"] - tab["cobertura_braco_fixo"],
])

plt.figure(figsize=(12, 6))
barras_sel = plt.bar(
    x - largura/2,
    tab["cobertura_empirica"],
    width=largura,
    yerr=erro_sel,
    capsize=4,
    label="Braço selecionado",
)
barras_fixo = plt.bar(
    x + largura/2,
    tab["cobertura_braco_fixo"],
    width=largura,
    yerr=erro_fixo,
    capsize=4,
    label="Braço fixado previamente",
)

plt.axhline(0.95, linestyle="--", linewidth=2, label="Cobertura nominal")
plt.xticks(x, ordem)
plt.ylim(0.80, 1.00)
plt.ylabel("Cobertura empírica")
plt.title("Cobertura do IC usual: efeito da seleção")
plt.legend()

for barras in [barras_sel, barras_fixo]:
    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura + 0.002,
            f"{altura:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Distribuição da média selecionada
# ------------------------------------------------------------------
plt.figure(figsize=(11, 6))
dados = [
    resultados1.loc[resultados1.politica == p, "media_selecionada"]
    for p in ordem
]
plt.boxplot(dados, tick_labels=ordem, showfliers=False)
plt.axhline(0, linestyle="--")
plt.ylabel("Média do braço selecionado")
plt.title("Viés do vencedor: distribuição da maior média amostral")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Diagnósticos adequados de concentração da alocação no caso nulo
# ------------------------------------------------------------------

# 1. Distribuição da proporção alocada ao braço mais escolhido.
dados_prop = [
    resultados1.loc[
        resultados1.politica == p,
        "proporcao_mais_alocada"
    ]
    for p in POLITICAS1
]

plt.figure(figsize=(11, 6))
plt.boxplot(dados_prop, tick_labels=POLITICAS1, showfliers=False)
plt.ylabel("Proporção destinada ao braço mais escolhido")
plt.title("Concentração da alocação dentro de cada experimento")
plt.ylim(0, 1.02)
plt.tight_layout()
plt.show()


# 2. Distribuição do tamanho amostral do braço selecionado.
dados_n_sel = [
    resultados1.loc[resultados1.politica == p, "n_selecionado"]
    for p in POLITICAS1
]

plt.figure(figsize=(11, 6))
plt.boxplot(dados_n_sel, tick_labels=POLITICAS1, showfliers=False)
plt.ylabel(r"$n_{\widehat a}$")
plt.title("Tamanho amostral do braço selecionado")
plt.tight_layout()
plt.show()


# 3. Contagens ordenadas: identidade do braço é removida.
cols_ordem = [f"n_ordem_{j}" for j in range(1, K1 + 1)]
alocacao_ordenada = (
    resultados1
    .groupby("politica", sort=False)[cols_ordem]
    .mean()
)

alocacao_ordenada.columns = [
    "Mais escolhido",
    "2º",
    "3º",
    "4º",
    "Menos escolhido",
]

display(alocacao_ordenada)

ax = alocacao_ordenada.T.plot(
    kind="bar",
    figsize=(12, 7),
    color=[CORES_POLITICAS[p] for p in alocacao_ordenada.index],
)
plt.axhline(T1 / K1, linestyle="--", linewidth=1.5, label="Alocação uniforme esperada")
plt.ylabel("Número médio de alocações")
plt.xlabel("Posição após ordenar as contagens")
plt.title("Alocação média ordenada dentro de cada experimento")
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()


# 4. Tabela-resumo de concentração.
resumo_concentracao1 = (
    resultados1
    .groupby("politica", sort=False)
    .agg(
        media_n_selecionado=("n_selecionado", "mean"),
        media_n_mais_alocado=("n_mais_alocado", "mean"),
        media_prop_mais_alocada=("proporcao_mais_alocada", "mean"),
    )
    .reset_index()
)

display(resumo_concentracao1)

## 1.3 Respostas conceituais

**1. A cobertura fica próxima de 95%?**  
Compare a tabela e o gráfico com 0,95. O intervalo usual ignora que o braço foi escolhido por apresentar a maior média. Essa seleção favorece estimativas altas por acaso. Mesmo a alocação uniforme está sujeita ao viés de seleção; nas políticas adaptativas, a própria quantidade de dados de cada braço também depende dos resultados anteriores.

**2. Como as políticas afetam $\widehat\mu_{\widehat a}$?**  
Como todas as médias verdadeiras são zero, uma distribuição deslocada para valores positivos é consequência da seleção do máximo. As políticas alteram a concentração das alocações, a persistência de vantagens iniciais aleatórias e a variabilidade da média vencedora.

**3. Relevância prática.**  
Em testes A/B, ensaios adaptativos e seleção de modelos, escolher a melhor alternativa e depois aplicar inferência convencional à mesma alternativa pode superestimar seu efeito, reduzir a cobertura dos intervalos e gerar valores-p excessivamente otimistas. Possíveis soluções incluem amostra confirmatória independente, correções por multiplicidade e inferência seletiva.

# Parte 2 — Bandit Poisson

Nesta parte, o objetivo muda: agora queremos comparar as políticas em termos de **recompensa acumulada**. A historinha é a de um site que escolhe diariamente uma entre cinco campanhas e observa o número de novos cadastros ao final do dia.

O modelo do enunciado é

$$
R_t\mid a_t=a \sim \text{Poisson}(\lambda_a),
$$

com

$$
\lambda=(18,20,21,23,27).
$$

Portanto, a campanha 5 é a melhor em média, pois possui a maior taxa esperada de cadastros.

O objetivo principal é comparar

$$
\sum_{t=1}^{T}R_t,
$$

isto é, o total de cadastros acumulados ao longo do experimento.

## 2.1 Priori e políticas comparadas na Parte 2

O enunciado pede prioris independentes

$$
\lambda_a\sim \text{Gamma}(\alpha_0,\beta_0),
$$

com parametrização por **taxa**. Usamos

$$
\alpha_0=4,
\qquad
\beta_0=0{,}2.
$$

Assim,

$$
E(\lambda_a)=\frac{\alpha_0}{\beta_0}=20,
\qquad
DP(\lambda_a)=\frac{\sqrt{\alpha_0}}{\beta_0}=10.
$$

Essa priori é centrada em 20 cadastros por dia, valor compatível com a escala das taxas verdadeiras, mas possui desvio-padrão 10, permitindo incerteza relevante antes da coleta dos dados.

Como Poisson e Gamma são conjugadas, após observar soma de recompensas $S_a$ e $n_a$ alocações no braço $a$,

$$
\lambda_a\mid D_t
\sim
\text{Gamma}(\alpha_0+S_a,\beta_0+n_a).
$$

A média posterior usada por algumas políticas é

$$
E(\lambda_a\mid D_t)
=
\frac{\alpha_0+S_a}{\beta_0+n_a}.
$$

As políticas comparadas são:

- **Alocação uniforme aleatória:** escolhe cada campanha com probabilidade $1/5$ em todos os dias.

- **Greedy:** escolhe a campanha com maior média posterior atual. Não possui exploração explícita.

- **Thompson Sampling:** sorteia

  $$
  \widetilde\lambda_a
  \sim
  \text{Gamma}(\alpha_a,\beta_a)
  $$

  para cada campanha e escolhe aquela com maior valor sorteado.

- **UCB:** escolhe a campanha com maior escore

  $$
  \text{escore}_a(t)
  =
  E(\lambda_a\mid D_t)
  +
  c\sqrt{\log(t+2)}
  \sqrt{\text{Var}(\lambda_a\mid D_t)}.
  $$

  Usamos $c=2$, mantendo a mesma lógica de bônus de incerteza usada na Parte 1.

- **Epsilon-greedy:** com probabilidade $1-\epsilon$, escolhe a campanha com maior média posterior; com probabilidade $\epsilon$, escolhe uma campanha aleatória. Usamos

  $$
  \epsilon=0{,}10.
  $$

  Assim, a política mantém 10% de exploração aleatória ao longo do experimento.

No código, a distribuição Gamma é parametrizada teoricamente por taxa $\beta$, mas a função `gamma` do NumPy usa **escala**. Por isso, ao simular de uma Gamma com taxa $\beta$, usamos `scale = 1 / beta`.

In [ ]:
# Parâmetros da Parte 2
LAMBDA_TRUE = np.array([18., 20., 21., 23., 27.])
K2 = len(LAMBDA_TRUE)
T2 = 500
ALFA0, BETA0 = 4.0, 0.2  # Gamma parametrizada por taxa
EPSILON = 0.10
C_UCB2 = 2.0

POLITICAS2 = [
    "Aleatória",
    "Greedy",
    "Thompson Sampling",
    "UCB",
    "Epsilon-Greedy",
]


def simular_poisson(
    politica,
    semente,
    horizonte=T2,
    alfa0=ALFA0,
    beta0=BETA0,
    inicializar_todos=False,
):
    """
    Simula um bandit Poisson-Gamma.

    Observação importante:
    - A teoria usa Gamma(alfa, beta), com beta como taxa.
    - O NumPy usa shape e scale; portanto, scale = 1 / beta.
    """
    gerador = np.random.default_rng(semente)

    contagens = np.zeros(K2, dtype=int)
    somas = np.zeros(K2)

    traj_recompensa = np.zeros(horizonte)
    traj_pseudo_arrependimento = np.zeros(horizonte)

    acumulado = 0.0
    pseudo_arrependimento = 0.0
    inicio = 0

    # Análise opcional: uma observação inicial por braço.
    if inicializar_todos:
        if horizonte < K2:
            raise ValueError("O horizonte deve ser ao menos igual ao número de braços.")

        for a in range(K2):
            r = gerador.poisson(LAMBDA_TRUE[a])

            contagens[a] += 1
            somas[a] += r

            acumulado += r
            pseudo_arrependimento += np.max(LAMBDA_TRUE) - LAMBDA_TRUE[a]

            traj_recompensa[a] = acumulado
            traj_pseudo_arrependimento[a] = pseudo_arrependimento

        inicio = K2

    for t in range(inicio, horizonte):
        alfa = alfa0 + somas
        beta = beta0 + contagens

        medias = alfa / beta
        desvios = np.sqrt(alfa / beta**2)

        if politica == "Aleatória":
            a = int(gerador.integers(K2))

        elif politica == "Greedy":
            a = argmax_aleatorio(medias, gerador)

        elif politica == "Thompson Sampling":
            amostras = gerador.gamma(shape=alfa, scale=1 / beta)
            a = argmax_aleatorio(amostras, gerador)

        elif politica == "UCB":
            beta_t = C_UCB2 * np.sqrt(np.log(t + 2))
            a = argmax_aleatorio(medias + beta_t * desvios, gerador)

        elif politica == "Epsilon-Greedy":
            if gerador.random() < EPSILON:
                a = int(gerador.integers(K2))
            else:
                a = argmax_aleatorio(medias, gerador)

        else:
            raise ValueError(f"Política desconhecida: {politica}")

        r = gerador.poisson(LAMBDA_TRUE[a])

        contagens[a] += 1
        somas[a] += r

        acumulado += r
        pseudo_arrependimento += np.max(LAMBDA_TRUE) - LAMBDA_TRUE[a]

        traj_recompensa[t] = acumulado
        traj_pseudo_arrependimento[t] = pseudo_arrependimento

    estimativas = (alfa0 + somas) / (beta0 + contagens)

    braco_otimo = int(np.argmax(LAMBDA_TRUE))
    braco_mais_alocado = int(np.argmax(contagens))
    braco_estimado_otimo = int(np.argmax(estimativas))

    return {
        "politica": politica,
        "trajetoria": traj_recompensa,
        "trajetoria_pseudo_arrependimento": traj_pseudo_arrependimento,
        "recompensa_final": traj_recompensa[-1],
        "pseudo_arrependimento_final": traj_pseudo_arrependimento[-1],
        "contagens": contagens,
        "estimativas": estimativas,
        "eqm_lambdas": np.mean((estimativas - LAMBDA_TRUE) ** 2),
        "eqm_braco_otimo": (estimativas[braco_otimo] - LAMBDA_TRUE[braco_otimo]) ** 2,
        "eqm_bracos_subotimos": np.mean(
            (
                estimativas[np.arange(K2) != braco_otimo]
                - LAMBDA_TRUE[np.arange(K2) != braco_otimo]
            ) ** 2
        ),
        "identificou_otimo": braco_estimado_otimo == braco_otimo,
        "braco_mais_alocado": braco_mais_alocado + 1,
        "mais_alocado_e_otimo": braco_mais_alocado == braco_otimo,
        "proporcao_no_otimo": contagens[braco_otimo] / horizonte,
        "proporcao_mais_alocada": np.max(contagens) / horizonte,
    }


# Teste limpo de uma única execução
resultado_teste_2 = simular_poisson(
    "Thompson Sampling",
    SEMENTE_GLOBAL
)

print("Teste de uma execução — Thompson Sampling")
print("Recompensa final:", resultado_teste_2["recompensa_final"])
print("Pseudo-arrependimento final:", resultado_teste_2["pseudo_arrependimento_final"])
print("Contagens por campanha:", resultado_teste_2["contagens"])
print("Estimativas finais de lambda:", np.round(resultado_teste_2["estimativas"], 4))
print("EQM médio dos lambdas:", round(resultado_teste_2["eqm_lambdas"], 4))
print("EQM do braço ótimo:", round(resultado_teste_2["eqm_braco_otimo"], 4))
print("EQM dos braços subótimos:", round(resultado_teste_2["eqm_bracos_subotimos"], 4))
print("Braço mais alocado:", resultado_teste_2["braco_mais_alocado"])
print("Mais alocado é o ótimo?", resultado_teste_2["mais_alocado_e_otimo"])

## 2.1 Simulação de Monte Carlo

In [ ]:
B2 = 1000 if MODO_COMPLETO else 150
trajetorias = {}
trajetorias_pseudo_arrependimento = {}
linhas2 = []

for i, politica in enumerate(POLITICAS2):
    mat_recompensa = np.zeros((B2, T2))
    mat_arrependimento = np.zeros((B2, T2))

    for b in tqdm(range(B2), desc=f"Parte 2 — {politica}"):
        res = simular_poisson(
            politica,
            SEMENTE_GLOBAL + 1_000_000 + 100_000 * i + b
        )

        mat_recompensa[b] = res["trajetoria"]
        mat_arrependimento[b] = res["trajetoria_pseudo_arrependimento"]

        linha = {
            "politica": politica,
            "repeticao": b,
            "recompensa_final": res["recompensa_final"],
            "pseudo_arrependimento_final": res["pseudo_arrependimento_final"],
            "eqm_lambdas": res["eqm_lambdas"],
            "eqm_braco_otimo": res["eqm_braco_otimo"],
            "eqm_bracos_subotimos": res["eqm_bracos_subotimos"],
            "identificou_otimo": res["identificou_otimo"],
            "braco_mais_alocado": res["braco_mais_alocado"],
            "mais_alocado_e_otimo": res["mais_alocado_e_otimo"],
            "proporcao_no_otimo": res["proporcao_no_otimo"],
            "proporcao_mais_alocada": res["proporcao_mais_alocada"],
        }

        for j, n in enumerate(res["contagens"], 1):
            linha[f"n_braco_{j}"] = n

        for j, e in enumerate(res["estimativas"], 1):
            linha[f"lambda_estimado_{j}"] = e

        linhas2.append(linha)

    trajetorias[politica] = mat_recompensa
    trajetorias_pseudo_arrependimento[politica] = mat_arrependimento

resultados2 = pd.DataFrame(linhas2)
display(resultados2.head())

## 2.2 Recompensa acumulada e pseudo-arrependimento

Além da recompensa acumulada observada, calculamos o **pseudo-arrependimento acumulado**. Ele mede, em termos esperados, quanto se perde por escolher uma campanha diferente da melhor campanha verdadeira.

Como a melhor campanha é a 5, com $\lambda_5=27$, o pseudo-arrependimento acumulado até o tempo $T$ é

$$
\sum_{t=1}^{T}
\left(
27-\lambda_{a_t}
\right).
$$

Essa medida não usa a recompensa sorteada em cada dia, mas sim as taxas verdadeiras da simulação. Por isso, ela mostra diretamente o custo esperado de explorar ou de ficar preso em campanhas subótimas.

In [ ]:
# ------------------------------------------------------------------
# Recompensa acumulada média e linha do oráculo
# ------------------------------------------------------------------
tempo = np.arange(1, T2 + 1)
recompensa_oraculo_esperada = np.max(LAMBDA_TRUE) * tempo

plt.figure(figsize=(12, 7))

for politica in POLITICAS2:
    mat = trajetorias[politica]
    media = mat.mean(axis=0)
    q05, q95 = np.quantile(mat, [.05, .95], axis=0)

    plt.plot(
        tempo,
        media,
        linewidth=2,
        label=politica,
        color=CORES_POLITICAS[politica],
    )
    plt.fill_between(
        tempo,
        q05,
        q95,
        alpha=.10,
        color=CORES_POLITICAS[politica],
    )

plt.plot(
    tempo,
    recompensa_oraculo_esperada,
    linestyle="--",
    linewidth=2,
    color="black",
    label="Oráculo: sempre campanha 5",
)

plt.xlabel("Dia")
plt.ylabel("Recompensa acumulada")
plt.title("Bandit Poisson: recompensa acumulada")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Pseudo-arrependimento acumulado
# ------------------------------------------------------------------
plt.figure(figsize=(12, 7))

for politica in POLITICAS2:
    mat = trajetorias_pseudo_arrependimento[politica]
    media = mat.mean(axis=0)
    q05, q95 = np.quantile(mat, [.05, .95], axis=0)

    plt.plot(
        tempo,
        media,
        linewidth=2,
        label=politica,
        color=CORES_POLITICAS[politica],
    )
    plt.fill_between(
        tempo,
        q05,
        q95,
        alpha=.10,
        color=CORES_POLITICAS[politica],
    )

plt.xlabel("Dia")
plt.ylabel("Pseudo-arrependimento acumulado")
plt.title("Custo esperado das escolhas subótimas")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Tabela-resumo
# ------------------------------------------------------------------
resumo2 = (
    resultados2
    .groupby("politica", sort=False)
    .agg(
        recompensa_media=("recompensa_final", "mean"),
        recompensa_dp=("recompensa_final", "std"),
        recompensa_q05=("recompensa_final", lambda x: x.quantile(.05)),
        recompensa_mediana=("recompensa_final", "median"),
        recompensa_q95=("recompensa_final", lambda x: x.quantile(.95)),
        pseudo_arrependimento_medio=("pseudo_arrependimento_final", "mean"),
        eqm_medio_lambdas=("eqm_lambdas", "mean"),
        eqm_medio_braco_otimo=("eqm_braco_otimo", "mean"),
        eqm_medio_bracos_subotimos=("eqm_bracos_subotimos", "mean"),
        prob_identificar_otimo=("identificou_otimo", "mean"),
        prob_mais_alocado_ser_otimo=("mais_alocado_e_otimo", "mean"),
        proporcao_media_no_otimo=("proporcao_no_otimo", "mean"),
        proporcao_media_mais_alocada=("proporcao_mais_alocada", "mean"),
    )
    .reset_index()
)

recompensa_oraculo_final = T2 * np.max(LAMBDA_TRUE)
resumo2["arrependimento_por_recompensa_media"] = (
    recompensa_oraculo_final - resumo2["recompensa_media"]
)
resumo2["percentual_do_oraculo"] = (
    100 * resumo2["recompensa_media"] / recompensa_oraculo_final
)

resumo2 = resumo2.sort_values("recompensa_media", ascending=False).reset_index(drop=True)
display(resumo2)

In [ ]:
# ------------------------------------------------------------------
# Distribuição da recompensa final, ordenada pelo desempenho médio
# ------------------------------------------------------------------
ordem_recompensa = resumo2["politica"].tolist()
dados = [
    resultados2.loc[
        resultados2.politica == p,
        "recompensa_final"
    ]
    for p in ordem_recompensa
]

plt.figure(figsize=(12, 6))
box = plt.boxplot(dados, tick_labels=ordem_recompensa, showfliers=False)

for posicao, politica in enumerate(ordem_recompensa, start=1):
    media = resultados2.loc[
        resultados2.politica == politica,
        "recompensa_final"
    ].mean()
    plt.scatter(posicao, media, marker="D", s=45, color=CORES_POLITICAS[politica])
    plt.text(
        posicao,
        media + 45,
        f"{media:,.0f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.axhline(
    T2 * np.max(LAMBDA_TRUE),
    linestyle="--",
    linewidth=1.5,
    color="black",
    label="Recompensa esperada do oráculo",
)
plt.ylabel("Recompensa acumulada final")
plt.title(f"Recompensa após {T2} dias")
plt.xticks(rotation=15)
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Alocação média por campanha
# ------------------------------------------------------------------
cols_n2 = [f"n_braco_{j}" for j in range(1, K2 + 1)]
aloc2 = resultados2.groupby("politica", sort=False)[cols_n2].mean()
aloc2 = aloc2.loc[POLITICAS2]
aloc2.columns = [f"Campanha {j}" for j in range(1, K2 + 1)]

display(aloc2)

aloc2.T.plot(
    kind="bar",
    figsize=(12, 7),
    color=[CORES_POLITICAS[p] for p in aloc2.index],
)
plt.axhline(T2, linestyle=":", linewidth=1, color="black")
plt.text(
    4.45,
    T2 - 7,
    "Total do horizonte = 500 dias",
    ha="right",
    va="top",
    fontsize=9,
)
plt.ylabel("Número médio de dias")
plt.title("Alocação média por campanha")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Instabilidade do greedy
# ------------------------------------------------------------------
dados_greedy = resultados2[resultados2["politica"] == "Greedy"]

plt.figure(figsize=(10, 5))
plt.hist(dados_greedy["recompensa_final"], bins=35, density=True, alpha=.75)
plt.axvline(
    dados_greedy["recompensa_final"].mean(),
    linestyle="--",
    linewidth=2,
    label=f"Média = {dados_greedy['recompensa_final'].mean():,.0f}",
)
plt.axvline(
    dados_greedy["recompensa_final"].median(),
    linestyle="-",
    linewidth=2,
    label=f"Mediana = {dados_greedy['recompensa_final'].median():,.0f}",
)
plt.xlabel("Recompensa acumulada final")
plt.ylabel("Densidade")
plt.title("Greedy: distribuição da recompensa final")
plt.legend()
plt.tight_layout()
plt.show()


prob_braço_dominante_greedy = (
    dados_greedy["braco_mais_alocado"]
    .value_counts(normalize=True)
    .sort_index()
    .rename_axis("campanha_mais_alocada")
    .reset_index(name="probabilidade")
)

recompensa_condicional_greedy = (
    dados_greedy
    .groupby("braco_mais_alocado", sort=True)
    .agg(
        numero_repeticoes=("repeticao", "size"),
        recompensa_media=("recompensa_final", "mean"),
        recompensa_mediana=("recompensa_final", "median"),
    )
    .reset_index()
    .rename(columns={"braco_mais_alocado": "campanha_mais_alocada"})
)

print("Probabilidade de cada campanha dominar a alocação do greedy:")
display(prob_braço_dominante_greedy)

print("Recompensa do greedy condicionada à campanha dominante:")
display(recompensa_condicional_greedy)

In [ ]:
# ------------------------------------------------------------------
# Exploração e concentração
# ------------------------------------------------------------------
# Entropia normalizada:
# 1 = distribuição próxima da uniforme;
# 0 = concentração extrema.

def entropia_linha(linha):
    n = linha[cols_n2].to_numpy(float)
    p = n / n.sum()
    p = p[p > 0]
    return -np.sum(p * np.log(p)) / np.log(K2)


resultados2["entropia_alocacao"] = resultados2.apply(
    entropia_linha,
    axis=1
)

exploracao = (
    resultados2
    .groupby("politica", sort=False)
    .agg(
        entropia_media=("entropia_alocacao", "mean"),
        media_dias_no_otimo=("n_braco_5", "mean"),
        proporcao_media_no_otimo=("proporcao_no_otimo", "mean"),
        proporcao_media_mais_alocada=("proporcao_mais_alocada", "mean"),
        prob_mais_alocado_ser_otimo=("mais_alocado_e_otimo", "mean"),
    )
    .reset_index()
)

display(exploracao.sort_values("entropia_media", ascending=False))

In [ ]:
# ------------------------------------------------------------------
# Qualidade de estimação de cada lambda
# ------------------------------------------------------------------
linhas = []

for politica in POLITICAS2:
    d = resultados2[resultados2.politica == politica]

    for j, verdadeiro in enumerate(LAMBDA_TRUE, 1):
        estimativas = d[f"lambda_estimado_{j}"]

        linhas.append({
            "politica": politica,
            "braco": j,
            "lambda_verdadeiro": verdadeiro,
            "media_estimativa": estimativas.mean(),
            "vies": estimativas.mean() - verdadeiro,
            "eqm": np.mean((estimativas - verdadeiro)**2),
            "n_medio": d[f"n_braco_{j}"].mean(),
        })

qualidade = pd.DataFrame(linhas)
display(qualidade)


# Mesmo ordenamento e mesmas cores em todos os gráficos.
tabela_eqm = (
    qualidade
    .pivot(index="braco", columns="politica", values="eqm")
    [POLITICAS2]
)

tabela_eqm.plot(
    kind="bar",
    figsize=(12, 7),
    color=[CORES_POLITICAS[p] for p in POLITICAS2],
)
plt.ylabel("Erro quadrático médio")
plt.xlabel("Braço")
plt.title("Erro quadrático médio das estimativas de cada taxa")
plt.text(
    0.01,
    0.97,
    "Valores menores indicam melhor estimação",
    transform=plt.gca().transAxes,
    ha="left",
    va="top",
    fontsize=10,
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Número médio de observações ao lado do EQM.
fig, ax = plt.subplots(figsize=(12, 7))

for politica in POLITICAS2:
    d = qualidade[qualidade["politica"] == politica]
    ax.scatter(
        d["n_medio"],
        d["eqm"],
        s=80,
        label=politica,
        color=CORES_POLITICAS[politica],
    )

    for _, linha in d.iterrows():
        ax.annotate(
            f"B{int(linha['braco'])}",
            (linha["n_medio"], linha["eqm"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

ax.set_xlabel("Número médio de observações no braço")
ax.set_ylabel("Erro quadrático médio")
ax.set_title("Relação entre quantidade de dados e erro de estimação")
ax.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Compromisso entre recompensa e estimação global
# ------------------------------------------------------------------
tabela_compromisso = resumo2[
    ["politica", "recompensa_media", "eqm_medio_lambdas"]
].copy()

display(tabela_compromisso)

plt.figure(figsize=(10, 6))

for _, linha in tabela_compromisso.iterrows():
    politica = linha["politica"]
    plt.scatter(
        linha["eqm_medio_lambdas"],
        linha["recompensa_media"],
        s=100,
        color=CORES_POLITICAS[politica],
    )
    plt.annotate(
        politica,
        (linha["eqm_medio_lambdas"], linha["recompensa_media"]),
        xytext=(6, 5),
        textcoords="offset points",
    )

plt.xlabel("EQM médio das cinco taxas")
plt.ylabel("Recompensa acumulada média")
plt.title("Compromisso entre recompensa e estimação global")
plt.tight_layout()
plt.show()

## 2.2 Análises de sensibilidade opcionais

As células abaixo são opcionais e ficam desativadas por padrão para não aumentar muito o tempo de execução.

Elas avaliam:

1. o greedy puro versus greedy após uma observação inicial por braço;
2. a sensibilidade do Thompson Sampling à escolha da priori Gamma.

Para executar, altere `EXECUTAR_ANALISES_EXTRAS` para `True` na primeira célula.

In [ ]:
if EXECUTAR_ANALISES_EXTRAS:
    B_EXTRA = 500 if MODO_COMPLETO else 100

    # --------------------------------------------------------------
    # Greedy puro versus greedy com rodada inicial
    # --------------------------------------------------------------
    linhas_init = []

    for inicializar_todos in [False, True]:
        rotulo = (
            "Greedy puro"
            if not inicializar_todos
            else "Greedy com 1 observação inicial por braço"
        )

        for b in tqdm(range(B_EXTRA), desc=rotulo):
            res = simular_poisson(
                "Greedy",
                SEMENTE_GLOBAL + 3_000_000
                + 100_000 * int(inicializar_todos)
                + b,
                inicializar_todos=inicializar_todos,
            )

            linhas_init.append({
                "versao": rotulo,
                "recompensa_final": res["recompensa_final"],
                "identificou_otimo": res["identificou_otimo"],
                "mais_alocado_e_otimo": res["mais_alocado_e_otimo"],
                "proporcao_no_otimo": res["proporcao_no_otimo"],
            })

    sensibilidade_inicializacao = pd.DataFrame(linhas_init)

    resumo_inicializacao = (
        sensibilidade_inicializacao
        .groupby("versao")
        .agg(
            recompensa_media=("recompensa_final", "mean"),
            recompensa_dp=("recompensa_final", "std"),
            prob_identificar_otimo=("identificou_otimo", "mean"),
            prob_mais_alocado_ser_otimo=("mais_alocado_e_otimo", "mean"),
            proporcao_media_no_otimo=("proporcao_no_otimo", "mean"),
        )
        .reset_index()
    )

    display(resumo_inicializacao)


    # --------------------------------------------------------------
    # Sensibilidade da priori no Thompson Sampling
    # Todas as prioris abaixo são parametrizadas por taxa.
    # --------------------------------------------------------------
    prioris = {
        "Atual: Gamma(4, 0.2), média 20": (4.0, 0.2),
        "Mais vaga: Gamma(1, 0.05), média 20": (1.0, 0.05),
        "Média 15: Gamma(3, 0.2)": (3.0, 0.2),
        "Média 25: Gamma(5, 0.2)": (5.0, 0.2),
    }

    linhas_priori = []

    for i, (rotulo, (alfa0, beta0)) in enumerate(prioris.items()):
        for b in tqdm(range(B_EXTRA), desc=rotulo):
            res = simular_poisson(
                "Thompson Sampling",
                SEMENTE_GLOBAL + 4_000_000 + 100_000 * i + b,
                alfa0=alfa0,
                beta0=beta0,
            )

            linhas_priori.append({
                "priori": rotulo,
                "recompensa_final": res["recompensa_final"],
                "pseudo_arrependimento_final": res["pseudo_arrependimento_final"],
                "eqm_lambdas": res["eqm_lambdas"],
                "proporcao_no_otimo": res["proporcao_no_otimo"],
            })

    sensibilidade_priori = pd.DataFrame(linhas_priori)

    resumo_priori = (
        sensibilidade_priori
        .groupby("priori")
        .agg(
            recompensa_media=("recompensa_final", "mean"),
            pseudo_arrependimento_medio=(
                "pseudo_arrependimento_final",
                "mean"
            ),
            eqm_medio=("eqm_lambdas", "mean"),
            proporcao_media_no_otimo=("proporcao_no_otimo", "mean"),
        )
        .reset_index()
    )

    display(resumo_priori)

else:
    print(
        "Análises extras não executadas. "
        "Altere EXECUTAR_ANALISES_EXTRAS para True para rodá-las."
    )

## 2.3 Respostas conceituais

**1. Quais políticas obtêm maior recompensa acumulada?**

A maior recompensa acumulada média foi obtida pelo **Thompson Sampling**, seguido por **UCB**, **epsilon-greedy**, **greedy** e, por último, pela alocação uniforme aleatória.

No experimento com $T=500$ dias, a recompensa esperada do oráculo, isto é, da política que já saberia desde o início que a campanha 5 é a melhor, seria

$$
500 \times 27 = 13.500.
$$

A partir da tabela `resumo2`, as políticas ficaram aproximadamente assim:

- Thompson Sampling: recompensa média de 13.355, cerca de 98,9% do oráculo;
- UCB: recompensa média de 13.260, cerca de 98,2% do oráculo;
- epsilon-greedy: recompensa média de 13.064, cerca de 96,8% do oráculo;
- greedy: recompensa média de 12.495, cerca de 92,6% do oráculo;
- aleatória: recompensa média de 10.901, cerca de 80,8% do oráculo.

Esse resultado indica que Thompson Sampling e UCB conseguiram identificar rapidamente que a campanha 5 era a melhor e concentraram nela a maior parte das decisões. A política aleatória teve desempenho inferior porque continua distribuindo as escolhas de forma quase uniforme entre todas as campanhas, inclusive as menos eficientes.

**2. Quais políticas exploram mais? Quais exploram menos?**

A política que mais explora é a **alocação uniforme aleatória**, pois escolhe as cinco campanhas com probabilidades aproximadamente iguais durante todo o experimento. Isso aparece na entropia média próxima de 1 e na alocação de cerca de 100 dias para cada campanha.

Entre as políticas adaptativas, a **epsilon-greedy** mantém exploração explícita, pois escolhe uma campanha aleatória com probabilidade $\epsilon=0{,}10$. Por isso, mesmo aprendendo que a campanha 5 é melhor, ela continua destinando parte dos dias às outras campanhas.

O **UCB** explora por meio do bônus de incerteza: campanhas pouco observadas recebem um aumento no escore, o que incentiva novas observações. O **Thompson Sampling** explora por amostragem posterior: campanhas incertas ainda podem gerar valores sorteados altos e, com isso, serem escolhidas.

A política **greedy** é a que menos explora, porque sempre escolhe a campanha com maior média posterior corrente. No entanto, baixa exploração não significa necessariamente boa concentração no braço ótimo. O greedy concentra quase todo o experimento em uma campanha, mas essa campanha pode ser subótima se tiver tido sorte no início.

**3. A política greedy é competitiva? Em que situações ela falha?**

A política greedy pode ser competitiva quando as primeiras observações favorecem corretamente a campanha 5. Nesses casos, ela passa a escolher a campanha ótima quase sempre e obtém recompensa próxima à do oráculo.

Porém, os resultados mostram que ela é muito instável. A recompensa média do greedy foi bem menor que sua mediana, e o desvio-padrão foi muito maior do que o das demais políticas adaptativas. Isso ocorre porque o greedy frequentemente fica preso em uma campanha subótima.

Na simulação, a campanha 5 foi a campanha mais alocada em apenas cerca de 57,6% das repetições. Em aproximadamente 30,8% das repetições, o greedy ficou preso na campanha 4; em 10,6%, na campanha 3; e em 1,0%, na campanha 2. Quando a campanha dominante foi a 5, a recompensa média foi alta. Quando a campanha dominante foi uma campanha inferior, a recompensa caiu bastante.

Portanto, o greedy falha quando uma campanha subótima apresenta recompensas iniciais altas por acaso, ou quando a campanha ótima apresenta recompensas iniciais baixas. Como a política não possui um mecanismo sistemático de exploração, ela pode não corrigir esse erro inicial.

**4. A política que maximiza recompensa acumulada também estima bem todos os $\lambda_a$?**

Não necessariamente. Os resultados mostram um conflito entre **maximizar recompensa acumulada** e **estimar bem todas as taxas**.

A política aleatória teve a menor recompensa acumulada, mas apresentou o menor erro quadrático médio global para as estimativas dos cinco $\lambda_a$. Isso acontece porque ela distribui as observações de forma equilibrada entre as campanhas, gerando informação sobre todas elas.

Por outro lado, o Thompson Sampling teve a maior recompensa acumulada, mas não estimou igualmente bem todos os parâmetros. Ele alocou em média cerca de 474 dos 500 dias à campanha 5. Com isso, estimou muito bem $\lambda_5$, mas coletou poucas observações das campanhas 1 a 4, gerando maior erro na estimação dos braços subótimos.

Assim, uma política pode ser excelente para decisão e recompensa, mas ruim para aprender todos os parâmetros com a mesma precisão. O objetivo de um bandit de recompensa é identificar e explorar rapidamente boas alternativas, não estimar todos os braços de forma uniforme.

**5. O modelo Poisson é adequado para o exemplo? Quais limitações ele tem?**

O modelo Poisson é um ponto de partida razoável porque a recompensa diária é uma contagem de novos cadastros, isto é, uma variável não negativa e discreta. Além disso, o parâmetro $\lambda_a$ tem interpretação direta como o número médio diário de cadastros gerados pela campanha $a$.

No entanto, o modelo é simplificado. Ele pressupõe que a média e a variância da contagem são iguais, que os dias são independentes, que a taxa de cada campanha é constante no tempo e que a exposição diária é comparável entre campanhas e dias.

Na prática, essas hipóteses podem falhar. O número de visitantes pode variar de um dia para outro, pode haver sazonalidade, diferenças por canal, perfil dos usuários, feriados, efeitos de repetição da campanha, custos diferentes e atraso entre exposição e cadastro. Também pode haver sobredispersão, isto é, variância maior que a média, o que não é bem capturado pelo modelo Poisson simples.

**6. Que mudanças no modelo seriam necessárias para tornar o exemplo mais realista?**

Algumas extensões naturais seriam:

- usar um modelo **Binomial** ou **Beta-Binomial** quando o número de visitantes expostos à campanha é conhecido;
- usar um modelo Poisson com **offset** para considerar o número de visitantes ou impressões;
- usar **Binomial negativa** para lidar com sobredispersão;
- considerar modelos com **inflação de zeros**, caso existam muitos dias sem cadastros;
- usar **bandits contextuais**, incorporando perfil do visitante, canal, dispositivo, região, horário e histórico de interação;
- incluir efeitos de calendário, como dia da semana, feriados, sazonalidade e tendência;
- permitir **não estacionariedade**, pois a eficácia de uma campanha pode mudar ao longo do tempo;
- incluir custos das campanhas e otimizar lucro ou cadastros líquidos, não apenas número bruto de cadastros;
- considerar recompensas atrasadas, pois alguns usuários podem se cadastrar dias depois de ver a campanha;
- considerar efeitos de saturação ou fadiga, quando usuários expostos repetidamente à mesma campanha passam a responder menos.

# 3. Verificações e exportação

In [ ]:
# Verificações de consistência
assert np.all(
    resultados1[[f"n_braco_{j}" for j in range(1, K1 + 1)]]
    .sum(axis=1) == T1
)

assert np.all(
    resultados2[[f"n_braco_{j}" for j in range(1, K2 + 1)]]
    .sum(axis=1) == T2
)

assert resumo1["cobertura_empirica"].between(0, 1).all()
assert resumo1["cobertura_braco_fixo"].between(0, 1).all()
assert (resultados2["recompensa_final"] >= 0).all()
assert resultados2["proporcao_no_otimo"].between(0, 1).all()
assert resultados2["proporcao_mais_alocada"].between(0, 1).all()

print("Todas as verificações foram aprovadas.")


# Arquivos úteis para o relatório
resumo1.to_csv(
    "resumo_parte1.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_concentracao1.to_csv(
    "concentracao_alocacao_parte1.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo2.to_csv(
    "resumo_parte2.csv",
    index=False,
    encoding="utf-8-sig"
)

exploracao.to_csv(
    "exploracao_parte2.csv",
    index=False,
    encoding="utf-8-sig"
)

qualidade.to_csv(
    "qualidade_estimacao_lambdas.csv",
    index=False,
    encoding="utf-8-sig"
)

if "prob_braço_dominante_greedy" in globals():
    prob_braço_dominante_greedy.to_csv(
        "probabilidade_campanha_dominante_greedy.csv",
        index=False,
        encoding="utf-8-sig"
    )

if "recompensa_condicional_greedy" in globals():
    recompensa_condicional_greedy.to_csv(
        "recompensa_greedy_por_campanha_dominante.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("Tabelas exportadas em CSV.")

# Conclusão

A Parte 1 mostra que selecionar a alternativa vencedora e aplicar um intervalo convencional à mesma estimativa pode comprometer a inferência. A Parte 2 mostra que maximizar recompensa e estimar igualmente bem todos os braços são objetivos diferentes. Políticas adaptativas precisam ser avaliadas tanto por desempenho decisório quanto pelas consequências estatísticas da coleta adaptativa.